# 40 — Train, track, and register

**Layer:** gold → model · **Engine:** PySpark for the read, single-node Python for the fit
**Tracking:** MLflow (Fabric's built-in tracking server)

### Why this notebook is not "in Spark"

Spark reads gold. Everything after that runs on the driver in pandas and scikit-learn, and
that is a deliberate engineering decision, not a shortcut:

* the labelled cohort is roughly 10⁵ rows and ~60 features — it fits in driver memory with
  room to spare, and distributing it would add shuffle cost to a workload that has none;
* **Explainable Boosting Machines have no Spark ML equivalent.** Neither does SHAP, nor
  `miceforest`. The interpretability requirement that the EU AI Act imposes on high-risk
  clinical decision support is precisely what rules out the distributed algorithms;
* SynapseML's `LightGBMClassifier` would distribute — and would give us a black box we
  could not defend to a regulator.

So: **Spark where the data is big, Python where the maths is interpretable.** Saying that
out loud on stage is a stronger position than pretending everything is a Spark job.

## 1. Configuration and experiment setup

In [ ]:
import mlflow, mlflow.sklearn
import numpy as np, pandas as pd

GOLD_TABLE       = "gold.knee_features"
EXPERIMENT       = "knee-poor-outcome"          # one experiment = one prediction problem
REGISTERED_MODEL = "knee-poor-outcome-ebm"      # what gets deployed
RANDOM_STATE     = 42
MCID_THRESHOLD   = 7

# Fabric wires MLflow to the workspace automatically; set_experiment creates the item.
mlflow.set_experiment(EXPERIMENT)

# Autologging is off. It logs the wrong things for a regulated model: sklearn's default
# metrics are accuracy-first, and it fires on every internal fit inside GridSearchCV,
# burying the runs you actually care about. Everything below is logged explicitly.
mlflow.autolog(disable=True)

print(f"Experiment: {EXPERIMENT}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

## 2. What to tag, and why

Tags are the searchable metadata on a run. Metrics answer *"how good?"*; **tags answer
*"what is this, who made it, and may we deploy it?"*** — which is the question an auditor
asks. In the Fabric ML experiment UI they become filter columns, so a tagging scheme you
can query is a tagging scheme that survives eight weeks of iteration.

The scheme below has four groups. Adopt it wholesale or trim it, but keep the four
*categories* — each answers a different regulatory question.

| Tag | Why it earns its place |
|---|---|
| **Identity — what is this run?** | |
| `model_family` | `ebm` / `random_forest` / `logistic_regression`. The first thing you filter on. |
| `model_role` | `candidate`, `baseline`, `challenger`, `champion`. Distinguishes the model you intend to ship from the one you keep for contrast on stage. |
| `interpretability` | `glassbox` / `post_hoc_shap` / `blackbox`. Under the AI Act this is a *deployment gate*, not a nice-to-have — make it queryable. |
| `calibrated` | `true` / `false`. An uncalibrated probability must never reach a clinician. |
| **Data — what was it trained on?** | |
| `gold_table`, `gold_table_version` | The Delta version. Turns "trained on gold" into "trained on gold as of version 7", which time travel can reconstruct exactly. |
| `feature_set_version` | Bump when the feature list changes. Without it, two runs with different features look comparable and are not. |
| `target_definition` | `oks_delta<=7`. The label rule is a clinical decision; encode it where a clinician can see it. |
| `cohort` | `knee_provider_2016_2019`. |
| **Governance — may we deploy it?** | |
| `intended_use` | `preop_risk_triage_decision_support`. The AI Act's intended-purpose declaration, attached to the artefact. |
| `risk_class` | `high_risk_eu_ai_act`. |
| `human_oversight` | `clinician_in_the_loop`. |
| `phi` | `false` — records are de-identified. States the data-protection posture explicitly. |
| `model_card` | Link or path to the model card for this version. |
| `approved_by`, `approved_on` | Set **after** review, never by the training run itself. An empty `approved_by` is a deployment blocker you can query for. |
| **Provenance — can we rebuild it?** | |
| `git_sha`, `notebook`, `run_by`, `fabric_workspace` | Reproducibility. `git_sha` is the one people skip and the one auditors ask for. |

**Params vs. tags vs. metrics:** hyperparameters → `log_param` (numeric, compared across
runs). Descriptive facts → `set_tag` (filtered, editable after the fact). Anything you
would plot → `log_metric`. Tags are mutable, params are not — which is exactly why
approval status is a tag.

In [ ]:
import subprocess, getpass, datetime, json

def _git_sha():
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"],
                                       stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        return "unknown"      # Fabric notebooks are not a git checkout unless synced

FEATURE_SET_VERSION = "v3"        # bump whenever FEATURE_COLUMNS below changes

# Read the exact Delta version we are about to train on — this is the reproducibility anchor.
gold_version = (spark.sql(f"DESCRIBE HISTORY {GOLD_TABLE}")
                     .selectExpr("max(version) AS v").first()["v"])

BASE_TAGS = {
    # --- data ---------------------------------------------------------------
    "gold_table":            GOLD_TABLE,
    "gold_table_version":    str(gold_version),
    "feature_set_version":   FEATURE_SET_VERSION,
    "target_definition":     f"oks_delta<={MCID_THRESHOLD} (below MCID)",
    "cohort":                "knee_provider_nhs_proms_2016_2019",
    # --- governance ---------------------------------------------------------
    "intended_use":          "preop_risk_triage_decision_support",
    "risk_class":            "high_risk_eu_ai_act",
    "human_oversight":       "clinician_in_the_loop",
    "phi":                   "false",
    "model_card":            "code/misc/OKS_T1_Prediction_Model_Card.md",
    "approved_by":           "pending",   # a named clinician replaces this
    "approved_on":           "pending",
    # --- provenance ---------------------------------------------------------
    "git_sha":               _git_sha(),
    "notebook":              "40_train_register",
    "run_by":                getpass.getuser(),
}

print(f"Training against {GOLD_TABLE} @ Delta version {gold_version}")
print(json.dumps(BASE_TAGS, indent=2))

## 3. Load gold and enforce the leakage guard

Two assertions before a single model is fitted. Both would be caught by a reviewer eventually;
catching them here means they never make it into a metric anyone quotes.

In [ ]:
gold = spark.table(GOLD_TABLE).filter("has_label = true").toPandas()
print(f"Labelled episodes: {len(gold):,}")

FEATURE_COLUMNS = [
    # demographics
    "age_band_grouped", "sex",
    # pre-op status
    "t0_assisted", "t0_assisted_by", "t0_symptom_period", "t0_previous_surgery",
    "living_arrangements_grouped", "t0_disability",
    # EQ-5D dimensions
    "t0_mobility", "t0_self_care", "t0_activity", "t0_discomfort", "t0_anxiety",
    # comorbidities
    "heart_disease", "high_bp", "stroke", "circulation", "lung_disease", "diabetes",
    "kidney_disease", "nervous_system", "liver_disease", "cancer", "depression",
    "arthritis", "comorbidity_count",
    # Oxford Knee Score
    "oks_t0_pain", "oks_t0_night_pain", "oks_t0_washing", "oks_t0_transport",
    "oks_t0_walking", "oks_t0_standing", "oks_t0_limping", "oks_t0_kneeling",
    "oks_t0_work", "oks_t0_confidence", "oks_t0_shopping", "oks_t0_stairs",
    "oks_t0_score", "oks_pain_subscale", "oks_function_subscale", "oks_adl_subscale",
    # provider context
    "university_hospital", "independent_hospital", "region", "year",
]

# `sex` and `age_band_grouped` are categorical because gold gives them an explicit
# "not_recorded" level rather than a NULL. They therefore arrive complete, MICE finds
# nothing to impute in them, and no protected characteristic is ever reconstructed.
CATEGORICAL_COLUMNS = ["year", "age_band_grouped", "sex",
                       "living_arrangements_grouped", "region"]

# GUARD 1 — no post-operative column may reach the feature matrix.
leaks = [c for c in FEATURE_COLUMNS if c.startswith("t1_") or "_t1_" in c or c in ("oks_delta", "oks_t1_score")]
assert not leaks, f"Post-operative leakage in FEATURE_COLUMNS: {leaks}"

# GUARD 2 — every declared feature must actually exist in gold.
absent = [c for c in FEATURE_COLUMNS if c not in gold.columns]
assert not absent, f"Declared features missing from gold: {absent}"

X_all = gold[FEATURE_COLUMNS].copy()
y_all = gold["outcome_label"].astype(int)

print(f"Features: {len(FEATURE_COLUMNS)}  |  poor-outcome rate: {y_all.mean():.1%}")

# GUARD 3 -- protected characteristics must arrive complete, never imputed.
for col in ["sex", "age_band_grouped"]:
    n_null = X_all[col].isna().sum()
    assert n_null == 0, (f"{col} has {n_null} NULLs -- gold should have given them a "
                         f"not_recorded level. MICE must not impute this column.")
print(f"Suppressed demographics carried as an explicit level: "
      f"{(X_all['sex'] == 'not_recorded').sum():,} rows")

## 4. Split first, fit transforms second

The split happens **before** imputation and encoding. `miceforest` builds a LightGBM model
per column with missing values — fit it on the full dataset and test-set information flows
into the training features. The kernel is fitted on train and then *applied* to test via
`impute_new_data`, which is the only honest ordering.

In [ ]:
from sklearn.model_selection import train_test_split

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=RANDOM_STATE, stratify=y_all
)
X_train_raw = X_train_raw.reset_index(drop=True)
X_test_raw  = X_test_raw.reset_index(drop=True)
y_train     = y_train.reset_index(drop=True)
y_test      = y_test.reset_index(drop=True)

print(f"train {len(X_train_raw):,}  |  test {len(X_test_raw):,}")
print(f"train positive rate {y_train.mean():.1%}  |  test positive rate {y_test.mean():.1%}")

## 5. The preprocessor — one object, logged with the model

MICE kernel + one-hot encoder are wrapped in a single class. This matters more than it
looks: at serving time the endpoint receives a *gold-shaped* row and has to reproduce this
transform byte-for-byte. Logging the fitted object as an MLflow artifact — rather than
re-deriving it in the app — is what guarantees training and serving agree.

**What MICE does not touch.** `sex` and `age_band_grouped` reach this point with an
explicit `not_recorded` level instead of NULLs, so `miceforest` finds nothing missing
and leaves them alone. That is deliberate: NHS suppressed those values to protect
confidentiality, and reconstructing them would undo a disclosure control and put a
guessed protected characteristic into a clinical risk score. Guard 3 in §3 asserts it.

In [ ]:
import re
import miceforest as mf
from sklearn.preprocessing import OneHotEncoder

# Delta rejects ' ,;{}()\n\t=' in column names, and one-hot encoding turns the category
# "60 to 69" into `age_band_grouped_60 to 69` -- which fails the write in §5b with
# DELTA_INVALID_CHARACTERS_IN_COLUMN_NAMES.
#
# The rename belongs here, not at the Delta write. The estimator is fitted on whatever
# names this class emits, so sanitising at the source keeps the encoded matrix, the
# logged MLflow signature and `ml.train_matrix` in agreement. Rename only on the way
# into Delta and 42/43 read the table back under different names than the model was
# fitted with, and sklearn rejects the frame with a feature-name mismatch.
_DELTA_INVALID = re.compile(r"[ ,;{}()\n\t=]+")


def delta_safe(name):
    """`age_band_grouped_60 to 69` -> `age_band_grouped_60_to_69`."""
    return _DELTA_INVALID.sub("_", str(name)).strip("_")


class ProMsPreprocessor:
    """MICE imputation + one-hot encoding, fitted on train, applied everywhere."""

    def __init__(self, categorical_columns, random_state=77, mice_iterations=5):
        self.categorical_columns = list(categorical_columns)
        self.random_state = random_state
        self.mice_iterations = mice_iterations
        self.kernel_ = None
        self.encoder_ = None
        self.feature_names_ = None

    def __getstate__(self):
        """Make the fitted kernel picklable.

        miceforest keeps its mean-matching predictions as float16
        (_PRE_LINK_DATATYPE) and its own __getstate__ serialises them with
        df.to_parquet(). Parquet has no float16 type, so a default kernel raises
        ArrowNotImplementedError on pickle -- but that history is exactly what
        impute_new_data() needs at serving time, so we cannot simply drop it
        (save_all_iterations_data=False makes the kernel picklable and useless).

        Casting to float32 satisfies Parquet, costs a few MB, and does not change
        the imputation: the values feed a KD-tree nearest-neighbour lookup.
        """
        if self.kernel_ is not None:
            for store in (self.kernel_.candidate_preds, self.kernel_.imputation_values):
                for col, df in list(store.items()):
                    if (df.dtypes == "float16").any():
                        store[col] = df.astype("float32")
        return self.__dict__.copy()

    def _as_categorical(self, X):
        X = X.copy().replace([np.inf, -np.inf], np.nan)
        for c in self.categorical_columns:
            X[c] = X[c].astype("category")
        return X

    def fit(self, X, y=None):
        Xc = self._as_categorical(X).reset_index(drop=True)
        self.kernel_ = mf.ImputationKernel(data=Xc, random_state=self.random_state)
        self.kernel_.mice(
            iterations=self.mice_iterations,
            variable_parameters={
                # Rare categories need larger leaves or LightGBM refuses to split.
                # Only min_data_in_leaf: min_child_samples is a LightGBM alias for it, and
                # miceforest collapses aliases to the canonical name -- passing both means
                # one silently overwrites the other.
                "age_band_grouped":            {"min_data_in_leaf": 30},
                "living_arrangements_grouped": {"min_data_in_leaf": 10},
            },
        )
        imputed = self.kernel_.complete_data(dataset=0)

        self.encoder_ = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
        self.encoder_.fit(imputed[self.categorical_columns])
        self._dtypes = imputed.dtypes.to_dict()
        self.feature_names_ = self._assemble(imputed).columns.tolist()
        return self

    def _assemble(self, imputed):
        numeric = imputed.drop(columns=self.categorical_columns)
        encoded = pd.DataFrame(
            self.encoder_.transform(imputed[self.categorical_columns]),
            columns=[delta_safe(c) for c in
                     self.encoder_.get_feature_names_out(self.categorical_columns)],
            index=imputed.index,
        )
        out = pd.concat([numeric, encoded], axis=1)
        # Two categories that differ only in a stripped character would collapse into one
        # column and silently merge two levels. Fail loudly instead.
        assert out.columns.is_unique, (
            f"delta_safe() collided columns: "
            f"{out.columns[out.columns.duplicated()].tolist()}"
        )
        return out

    def transform(self, X):
        Xc = self._as_categorical(X).reset_index(drop=True)
        Xc = Xc.astype({k: v for k, v in self._dtypes.items() if k in Xc.columns})
        imputed = self.kernel_.impute_new_data(Xc).complete_data(0)
        out = self._assemble(imputed)
        # Serving may hand us a single row missing a rare category — align to training shape.
        return out.reindex(columns=self.feature_names_, fill_value=0)

    def fit_transform(self, X, y=None):
        return self.fit(X, y).transform(X)


pre = ProMsPreprocessor(CATEGORICAL_COLUMNS)
X_train = pre.fit_transform(X_train_raw)
X_test  = pre.transform(X_test_raw)

print(f"Encoded matrix: {X_train.shape[1]} features")
print(X_train.columns.tolist())

## 5b. Persist the split so every downstream notebook reuses it

`41_model_search`, `42_threshold_analysis` and `43_explainability` all need this exact
train/test split and this exact encoding. If each of them re-split and re-fitted MICE, the
numbers would not be comparable — different folds, different imputed values, and a "model
comparison" that compares nothing.

So the split is written once, here, and read everywhere else.

**These tables deliberately live in an `ml` schema, not in `gold`.** They contain
train-fitted transforms (MICE, one-hot), which is exactly what §5 argued must never enter
a shared contract table. `ml.*` is model-internal scratch: reproducible from gold plus the
logged preprocessor, owned by the training pipeline, and not something the app or the Data
Agent should ever read.

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS ml")

def _persist(X, y, table):
    pdf = X.copy()
    # Names are already Delta-safe: §5 sanitises them as the encoder emits them. str()
    # only covers the non-string column labels pandas allows.
    pdf.columns = [str(c) for c in pdf.columns]
    pdf["outcome_label"] = y.values
    (spark.createDataFrame(pdf)
          .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
          .saveAsTable(table))
    print(f"{table}: {len(pdf):,} rows x {pdf.shape[1]} cols")

_persist(X_train, y_train, "ml.train_matrix")
_persist(X_test,  y_test,  "ml.test_matrix")

# The pre-encoding gold rows for the same split — 43_explainability needs readable values
# to label a SHAP plot with "70 to 79" rather than "age_band_grouped_70 to 79 = 1".
for raw, name in [(X_train_raw, "ml.train_raw"), (X_test_raw, "ml.test_raw")]:
    (spark.createDataFrame(raw.astype({c: "str" for c in CATEGORICAL_COLUMNS}))
          .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
          .saveAsTable(name))

spark.sql("COMMENT ON TABLE ml.train_matrix IS "
          "'Model-internal. Encoded training matrix for one specific split, produced by "
          "40_train_register. Contains train-fitted MICE and one-hot output - not a shared "
          "contract table, do not read from the app or the Data Agent.'")
print("\nSplit persisted. 41/42/43 read these tables.")

## 6. Metrics: precision–recall, not accuracy

Roughly one patient in five has a poor outcome. A model that predicts "good outcome"
for everybody scores ~80% accuracy and identifies nobody who needs pre-operative
optimisation. Average precision (area under the PR curve) is the headline number because
it is the one that degrades when the minority class is ignored.

`brier_score` and `ece` (expected calibration error) are logged alongside because a
*probability* shown to a clinician has to mean what it says. A model can rank patients
perfectly and still be badly calibrated.

In [ ]:
from sklearn.metrics import (average_precision_score, roc_auc_score, brier_score_loss,
                             precision_score, recall_score, f1_score, precision_recall_curve)

def expected_calibration_error(y_true, y_prob, n_bins=10):
    """Mean absolute gap between predicted probability and observed frequency."""
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.digitize(y_prob, bins[1:-1])
    ece = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.sum() == 0:
            continue
        ece += (m.sum() / len(y_prob)) * abs(y_true[m].mean() - y_prob[m].mean())
    return float(ece)

def recall_at_precision(y_true, y_prob, target_precision=0.80):
    """Highest recall achievable while holding precision at the target. Returns (recall, threshold)."""
    p, r, t = precision_recall_curve(y_true, y_prob)
    ok = p[:-1] >= target_precision
    if not ok.any():
        return 0.0, 1.0
    i = int(np.argmax(r[:-1] * ok))
    return float(r[i]), float(t[i])

def evaluate(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true)
    y_hat = (y_prob >= threshold).astype(int)
    rec80, thr80 = recall_at_precision(y_true, y_prob, 0.80)
    return {
        "average_precision":  float(average_precision_score(y_true, y_prob)),   # PRIMARY
        "roc_auc":            float(roc_auc_score(y_true, y_prob)),
        "brier_score":        float(brier_score_loss(y_true, y_prob)),
        "ece":                expected_calibration_error(y_true, y_prob),
        "precision_at_0.5":   float(precision_score(y_true, y_hat, zero_division=0)),
        "recall_at_0.5":      float(recall_score(y_true, y_hat, zero_division=0)),
        "f1_at_0.5":          float(f1_score(y_true, y_hat, zero_division=0)),
        "recall_at_p80":      rec80,
        "threshold_for_p80":  thr80,
        "positive_rate":      float(y_true.mean()),
    }

BASELINE_AP = float(y_test.mean())    # what a coin weighted to the prevalence would score
print(f"No-skill average precision (prevalence): {BASELINE_AP:.3f} — every model must beat this.")

## 7. Train the candidates as nested runs

One **parent run** for the training session, one **child run** per model. This keeps the
experiment UI readable: collapse the parent to compare sessions, expand it to compare
models within a session.

Logistic regression is not there to win. It is the honest floor — if a glassbox EBM cannot
beat a linear model, the extra complexity is not earning anything.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from interpret.glassbox import ExplainableBoostingClassifier
import mlflow.models

# Declare integer columns as float64. At serving time a single missing value turns an
# int column into a float and MLflow schema enforcement rejects the request -- the
# warning MLflow emits here is describing a production failure, not a cosmetic issue.
signature = mlflow.models.infer_signature(
    X_train.astype("float64"),
    np.zeros(len(X_train), dtype=int),
)

CANDIDATES = {
    "logistic_regression": dict(
        estimator=Pipeline([("scale", StandardScaler()),
                            ("model", LogisticRegression(max_iter=2000, class_weight="balanced",
                                                         random_state=RANDOM_STATE))]),
        tags={"model_family": "logistic_regression", "model_role": "baseline",
              "interpretability": "glassbox", "calibrated": "false"},
    ),
    "random_forest": dict(
        estimator=RandomForestClassifier(n_estimators=400, max_depth=12, min_samples_leaf=20,
                                         class_weight="balanced_subsample", n_jobs=-1,
                                         random_state=RANDOM_STATE),
        tags={"model_family": "random_forest", "model_role": "challenger",
              "interpretability": "post_hoc_shap", "calibrated": "false"},
    ),
    "ebm": dict(
        estimator=ExplainableBoostingClassifier(interactions=10, outer_bags=8,
                                                random_state=RANDOM_STATE),
        tags={"model_family": "ebm", "model_role": "candidate",
              "interpretability": "glassbox", "calibrated": "false"},
    ),
}

results = {}
fitted  = {}

with mlflow.start_run(run_name="training-session") as parent:
    PARENT_RUN_ID = parent.info.run_id
    mlflow.set_tags({**BASE_TAGS, "run_type": "training_session"})
    mlflow.log_param("n_train", len(X_train))
    mlflow.log_param("n_test", len(X_test))
    mlflow.log_param("n_features", X_train.shape[1])
    mlflow.log_metric("baseline_average_precision", BASELINE_AP)

    for name, spec in CANDIDATES.items():
        with mlflow.start_run(run_name=name, nested=True):
            mlflow.set_tags({**BASE_TAGS, **spec["tags"], "run_type": "candidate"})

            est = spec["estimator"]
            est.fit(X_train, y_train)
            prob = est.predict_proba(X_test)[:, 1]
            metrics = evaluate(y_test, prob)

            mlflow.log_params({f"hp_{k}": v for k, v in est.get_params(deep=False).items()
                               if isinstance(v, (int, float, str, bool, type(None)))})
            mlflow.log_metrics(metrics)
            mlflow.log_metric("lift_over_baseline", metrics["average_precision"] / BASELINE_AP)
            mlflow.sklearn.log_model(est, artifact_path="model", signature=signature)

            results[name] = metrics
            fitted[name]  = est
            print(f"{name:22s} AP={metrics['average_precision']:.3f}  "
                  f"ROC-AUC={metrics['roc_auc']:.3f}  ECE={metrics['ece']:.3f}  "
                  f"recall@P80={metrics['recall_at_p80']:.3f}")

pd.DataFrame(results).T.round(4)

## 8. Calibrate the EBM — the run that ships

Ranking is not enough. A clinician told "34% risk" will act on the number, so the number
has to be right in aggregate: of a hundred patients scored 34%, about thirty-four should
have a poor outcome. Isotonic regression on a held-out fold fixes systematic over- or
under-confidence.

Watch `brier_score` and `ece` move, not `average_precision` — calibration is monotone, so
ranking metrics barely change. That is the point.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

with mlflow.start_run(run_name="ebm-calibrated") as cal_run:
    CALIBRATED_RUN_ID = cal_run.info.run_id
    mlflow.set_tags({
        **BASE_TAGS,
        "model_family":     "ebm",
        "model_role":       "champion",          # this is the one we intend to deploy
        "interpretability": "glassbox",
        "calibrated":       "true",
        "calibration_method": "isotonic",
        "run_type":         "release_candidate",
        "parent_run_id":    PARENT_RUN_ID,
    })

    calibrated = CalibratedClassifierCV(
        ExplainableBoostingClassifier(interactions=10, outer_bags=8, random_state=RANDOM_STATE),
        method="isotonic", cv=5,
    )
    calibrated.fit(X_train, y_train)

    prob_cal = calibrated.predict_proba(X_test)[:, 1]
    prob_raw = fitted["ebm"].predict_proba(X_test)[:, 1]

    m_cal = evaluate(y_test, prob_cal)
    mlflow.log_metrics(m_cal)
    mlflow.log_metric("brier_improvement", results["ebm"]["brier_score"] - m_cal["brier_score"])
    mlflow.log_metric("ece_improvement",   results["ebm"]["ece"]         - m_cal["ece"])
    mlflow.log_params({"calibration_cv": 5, "calibration_method": "isotonic",
                       "n_train": len(X_train), "n_features": X_train.shape[1]})

    # --- calibration plot, logged as an artifact -----------------------------
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfectly calibrated")
    for prob, label in [(prob_raw, "EBM (raw)"), (prob_cal, "EBM (isotonic)")]:
        t, p = calibration_curve(y_test, prob, n_bins=10, strategy="quantile")
        ax.plot(p, t, "o-", label=label)
    ax.set_xlabel("Predicted probability of poor outcome")
    ax.set_ylabel("Observed frequency")
    ax.set_title("Calibration — does 34% mean 34%?")
    ax.legend(); ax.grid(alpha=.3)
    mlflow.log_figure(fig, "calibration_curve.png")
    plt.show()

    # --- the preprocessor travels with the model ----------------------------
    import joblib, tempfile, os
    with tempfile.TemporaryDirectory() as d:
        pp = os.path.join(d, "preprocessor.joblib")
        joblib.dump(pre, pp)
        mlflow.log_artifact(pp, artifact_path="preprocessing")
        fp = os.path.join(d, "feature_columns.json")
        with open(fp, "w") as f:
            json.dump({"gold_columns": FEATURE_COLUMNS,
                       "categorical_columns": CATEGORICAL_COLUMNS,
                       "encoded_columns": pre.feature_names_}, f, indent=2)
        mlflow.log_artifact(fp, artifact_path="preprocessing")

    mlflow.sklearn.log_model(calibrated, artifact_path="model", signature=signature)

    print(f"raw  -> Brier {results['ebm']['brier_score']:.4f}  ECE {results['ebm']['ece']:.4f}")
    print(f"cal  -> Brier {m_cal['brier_score']:.4f}  ECE {m_cal['ece']:.4f}")
    print(f"AP unchanged: {results['ebm']['average_precision']:.3f} -> {m_cal['average_precision']:.3f}")

## 9. Register as a Fabric ML model

`log_model` records an experiment artifact. `register_model` promotes it to a **Fabric ML
model item** — a versioned, permissioned object in the workspace with its own lineage,
which is what the endpoint and the app bind to.

The distinction is the "no `model.pkl` emailed around" line from the session: the app
references `models:/knee-poor-outcome-ebm@champion`, never a file path.

In [ ]:
from mlflow import MlflowClient

client = MlflowClient()

mv = mlflow.register_model(
    model_uri=f"runs:/{CALIBRATED_RUN_ID}/model",
    name=REGISTERED_MODEL,
)
print(f"Registered {REGISTERED_MODEL} version {mv.version}")

# --- version-level tags: what the deployment gate reads ---------------------
# Fabric's model registry rejects empty tag values, so "not yet approved" needs an
# explicit sentinel. It reads better than "" anyway, which is ambiguous between
# "never set" and "deliberately cleared".
UNAPPROVED = "pending"

version_tags = {
    "calibrated":          "true",
    "interpretability":    "glassbox",
    "intended_use":        "preop_risk_triage_decision_support",
    "risk_class":          "high_risk_eu_ai_act",
    "human_oversight":     "clinician_in_the_loop",
    "gold_table_version":  str(gold_version),
    "feature_set_version": FEATURE_SET_VERSION,
    "git_sha":             BASE_TAGS["git_sha"] or "unknown",
    "average_precision":   f"{m_cal['average_precision']:.4f}",
    "ece":                 f"{m_cal['ece']:.4f}",
    "approved_by":         UNAPPROVED,   # a named clinician replaces this after review
    "approved_on":         UNAPPROVED,
}

for k, v in version_tags.items():
    v = str(v).strip()
    if not v:
        print(f"  skipped empty tag: {k}")
        continue
    client.set_model_version_tag(REGISTERED_MODEL, mv.version, k, v)

print(f"Tagged {REGISTERED_MODEL} v{mv.version} with {len(version_tags)} tags.")

client.update_model_version(
    name=REGISTERED_MODEL, version=mv.version,
    description=(
        f"Calibrated Explainable Boosting Machine predicting poor outcome "
        f"(OKS gain <= {MCID_THRESHOLD}) after primary knee replacement. "
        f"Trained on {GOLD_TABLE} @ Delta v{gold_version}, {len(X_train):,} episodes, "
        f"{X_train.shape[1]} features. Test average precision {m_cal['average_precision']:.3f} "
        f"vs. {BASELINE_AP:.3f} prevalence baseline; ECE {m_cal['ece']:.4f}. "
        f"Decision support only — requires clinician review."
    ),
)

CHAMPION_TAG = "champion"

def promote_to_champion(client, name, version):
    """Mark the version we intend to deploy.

    Aliases would be the right mechanism -- the app pins a label and never learns a
    version number. But Fabric's MLflow registry does not implement the alias
    endpoints (/api/2.0/mlflow/registered-models/alias returns 404), so we fall back
    to a version tag: exactly one version carries champion='true'. The alias is still
    attempted first, so this stays correct on a tenant where aliases do exist.
    """
    try:
        client.set_registered_model_alias(name, "champion", version)
        print(f"Alias @champion -> version {version}")
        return
    except Exception as e:
        print(f"Aliases unsupported here ({type(e).__name__}); using a version tag.")

    for other in client.search_model_versions(f"name='{name}'"):
        if str(other.version) != str(version) and other.tags.get(CHAMPION_TAG) == "true":
            client.set_model_version_tag(name, other.version, CHAMPION_TAG, "false")
    client.set_model_version_tag(name, version, CHAMPION_TAG, "true")
    print(f"champion=true -> version {version}")

promote_to_champion(client, REGISTERED_MODEL, mv.version)
print(f"Load with: mlflow.sklearn.load_model('models:/{REGISTERED_MODEL}/{mv.version}')")

## 10. Deployment gate

The last cell of a training notebook should be able to say **no**. These four checks are
the difference between a model registry and a folder of pickles.

In [ ]:
GATES = {
    "beats prevalence baseline by 1.3x": m_cal["average_precision"] >= BASELINE_AP * 1.3,
    "calibration error under 5%":        m_cal["ece"] <= 0.05,
    "recall at 80% precision above 5%":  m_cal["recall_at_p80"] >= 0.05,
    "interpretable model family":        True,   # EBM is glassbox by construction
}

for check, passed in GATES.items():
    print(f"  {'PASS' if passed else 'FAIL'}  {check}")

if all(GATES.values()):
    client.set_model_version_tag(REGISTERED_MODEL, mv.version, "gate_status", "passed")
    print(f"\nVersion {mv.version} passes automated gates. "
          f"Still requires a named clinical reviewer before `approved_by` is set.")
else:
    client.set_model_version_tag(REGISTERED_MODEL, mv.version, "gate_status", "failed")
    print("\nGates failed — do not promote.")

## 11. Query the experiment the way an auditor would

If the tagging scheme is right, these two queries answer "what is deployable and why"
without opening a single notebook.

In [ ]:
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT],
    filter_string="tags.risk_class = 'high_risk_eu_ai_act' and tags.calibrated = 'true'",
    order_by=["metrics.average_precision DESC"],
)
cols = [c for c in ["run_id", "tags.model_family", "tags.model_role", "tags.gold_table_version",
                    "metrics.average_precision", "metrics.ece", "tags.approved_by"] if c in runs.columns]
display(runs[cols].head(10))

print("\nAwaiting clinical sign-off:")
pending = mlflow.search_runs(
    experiment_names=[EXPERIMENT],
    filter_string="tags.run_type = 'release_candidate' and tags.approved_by = 'pending'",
)
cols = [c for c in ["run_id", "start_time", "metrics.average_precision"] if c in pending.columns]
display(pending[cols] if len(pending) else "none — every release candidate has been signed off")

---

**Next:** `50_batch_score` — score the pre-op cohort with `@champion`, compute per-patient
explanations, and write both back to gold for the clinician app and the Data Agent.